# Module 03 — AI Workbook Companion (Streams & Profiling)

A lightweight companion to
[Streaming and Visual Profiling.ipynb](../Streaming%20and%20Visual%20Profiling.ipynb).
It does **not** replace the existing exercises — it adds the supervision layer
from [Module 11](../../11/README.md) to Module 03's topic: **CUDA streams and
overlapping host↔device transfer with compute.**

New here? Read [Module 11's README](../../11/README.md) first — it explains the
5-phase loop, what a "CPU reference" is, and the known ways AI gets CUDA wrong.

## The loop, applied to streamed, chunked processing

Your task: process a large array in chunks, overlapping each chunk's copy with
another chunk's compute using multiple streams. You may have an AI write it. Your
job is everything around it.

1. **SPECIFY** — Contract: chunk count and size, one stream per in-flight chunk,
   pinned host memory for true async copies, the exact stream each H2D / kernel /
   D2H must run on, and the metric (total time vs the non-overlapped baseline).
2. **GENERATE** — Ask the AI for the streamed pipeline. Keep it in its own file.
3. **PREDICT** — *Before running:* will the timeline actually overlap, or will a
   hidden dependency serialize it? Which operation on which stream is the
   correctness risk? (Cross-stream ordering is the classic trap.)
4. **VERIFY** — Use [verify_streams.cu](./verify_streams.cu): CPU reference plus a
   harness *stub*. You complete the streamed launches and the per-stream
   synchronization, then check the **whole** array.
5. **DIAGNOSE** — Profile with `nsys`. Look at the timeline: are copies and
   kernels actually overlapping across streams, or stacked end-to-end?

## The adversarial exercise

[adversarial_streams_buggy.cu](./adversarial_streams_buggy.cu) is a streamed
pipeline "an AI wrote for you." It compiles and often prints the right answer.
**It has a cross-stream race:** each chunk's compute runs on its own stream, but
every chunk's device→host copy is issued on a *different* stream, so some copies
can read results before the matching kernel has finished.

Your job:

1. Read it and predict which chunks are at risk *before* running.
2. Build and run it several times. The failure may be intermittent — that is the
   lesson about stream ordering.
3. State the exact bug (which operation is on the wrong stream) and the fix.
4. Prove the fix with a full-array check across several runs.

Could an AI catch this? Sometimes — but a cross-stream ordering bug produces no
compiler error and often the right answer, so it slips past a single run and past
a casual review. A full-array check plus a profiler timeline catches it. That is
your job.

A worked solution and explanation are in [solution.ipynb](./solution.ipynb). Try it
yourself first.

## Reflection prompt (write this down)

- Which stream did each H2D, kernel, and D2H run on, and where did the ordering
  break? Why did the program still often print the right answer?
- In the `nsys` timeline, did the copies overlap the kernels? If not, what
  dependency serialized them (pageable vs pinned memory, default-stream
  synchronization, too few streams)?
- If the pipeline is transfer-bound, how much speedup can overlap actually buy,
  and how did you estimate that ceiling before measuring?


In [ ]:
# Prerequisites: verify the GPU toolchain before running this workbook.
import pathlib, runpy
_here = pathlib.Path.cwd()
for _d in (_here, *_here.parents):
    if (_d / "check_gpu.py").exists():
        runpy.run_path(str(_d / "check_gpu.py"), run_name="__main__")
        break
else:
    print("check_gpu.py not found - run `python check_gpu.py` from the repo root.")

## Step 1 - Reproduce the bug

Build and run the problem program [adversarial_streams_buggy.cu](./adversarial_streams_buggy.cu). Read it first and predict what will happen before you run this cell.

In [ ]:
!nvcc -arch=native adversarial_streams_buggy.cu -o buggy && ./buggy

## Step 2 - Run your verification harness

[verify_streams.cu](./verify_streams.cu) ships a CPU reference and a PASS/FAIL gate, but it **defaults to FAIL** until you complete the `TODO` sections in it (launch, error check, synchronization). Edit the file, then re-run this cell until it prints `PASS`.

In [ ]:
!nvcc -arch=native verify_streams.cu -o verify && ./verify

## Next

Once your harness prints `PASS`, open **[solution.ipynb](./solution.ipynb)** to compare your fix with the worked answer and explanation. See [Module 11](../../11/README.md) for the method behind this workbook.